# APML + MES — Live training viewer (JupyterLab)

Run the training command in a **separate JupyterLab Terminal** from the repository root, for example `python scripts/cluster_movie.py train --dataset mnist --out runs/mnist_movie_seed42 --video`. Then execute the cell below in this notebook. It watches only local files: no new port, no server permissions, no synthetic animation. Use *Interrupt Kernel* to stop watching; it does not stop the training process.


In [ ]:
from pathlib import Path
import json, time
from IPython.display import Image, display, clear_output

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent  # notebook opened from examples/
RUN = ROOT / "runs/mnist_movie_seed42"  # change if your --out differs
image_path = RUN / "live/latest.png"
config_path = RUN / "config.json"
print("Monitoring:", RUN)
last_seen = None
last_change = time.monotonic()
while True:
    if image_path.exists():
        stamp = image_path.stat().st_mtime_ns
        if stamp != last_seen:
            last_seen = stamp
            last_change = time.monotonic()
            # Read fresh bytes each time (bypasses image caching).
            clear_output(wait=True)
            display(Image(data=image_path.read_bytes()))
    status = None
    if config_path.exists():
        try:
            status = json.loads(config_path.read_text(encoding="utf-8")).get("status")
        except json.JSONDecodeError:
            pass
    if status in ("complete", "interrupted"):
        print(f"Training status: {status}. Movie: {RUN / 'movie.mp4'}")
        break
    if time.monotonic() - last_change > 1800:
        print("No updated snapshot for 30 minutes; stopping viewer. Training is unaffected.")
        break
    time.sleep(2)
